<a href="https://colab.research.google.com/github/fabrito2006/Laboratorio-7-MID/blob/develop/Laboratorio_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 05 - Mineria de Datos
## Alumno: Torres Alvaradao Fabricio Ismael
## Ciclo y carrera: 5C28A

## 1. Con base a datos de cáncer de mama alojados en el repositorio UCI Machine Learning Repository (https://archive.ics.uci.edu/ml/datasets/breast+cancer+wisconsin+%28original%29) y utilizando las librerías de Python que se indican, haga lo siguiente:

In [1]:
!pip install ucimlrepo

In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

# fetch dataset
breast_cancer_wisconsin_original = fetch_ucirepo(id=15)

# data (as pandas dataframes)
X = breast_cancer_wisconsin_original.data.features
y = breast_cancer_wisconsin_original.data.targets

# Concat data
data = pd.concat([X, y], axis=1)

### a . Calcule el information value (IV), tanto para el grupo de variable numéricas como categóricas y excluya las que tenga un poder predictivo débil o menor. Además, separe la variable de clasificación del resto de variables para luego obtener los datos de entrenamiento y prueba, tomando de este último el 25% de datos.

In [4]:
data.head(10)

,Clump_thickness,Uniformity_of_cell_size,Uniformity_of_cell_shape,Marginal_adhesion,Single_epithelial_cell_size,Bare_nuclei,Bland_chromatin,Normal_nucleoli,Mitoses,Class
0,5,1,1,1,2,1.0,3,1,1,2
1,5,4,4,5,7,10.0,3,2,1,2
2,3,1,1,1,2,2.0,3,1,1,2
3,6,8,8,1,3,4.0,3,7,1,2
4,4,1,1,3,2,1.0,3,1,1,2
5,8,10,10,8,7,10.0,9,7,1,4
6,1,1,1,1,2,10.0,3,1,1,2
7,2,1,2,1,2,1.0,3,1,1,2
8,2,1,1,1,2,1.0,1,1,5,2
9,4,2,1,1,2,1.0,2,1,1,2


In [21]:
# Limpiamos los valores nulos
data = data.dropna()

# Verificamos los valores nulos
data.isnull().sum()

,0
Clump_thickness,0
Uniformity_of_cell_size,0
Uniformity_of_cell_shape,0
Marginal_adhesion,0
Single_epithelial_cell_size,0
Bare_nuclei,0
Bland_chromatin,0
Normal_nucleoli,0
Mitoses,0
Class,0


In [3]:
!pip install feature_engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.6/378.6 kB 5.1 MB/s eta 0:00:00


In [13]:
# Separamos las variables númericas
v_numericas = data.select_dtypes(include=['int64', 'float64']).columns
# Eliminamos la variable class
v_numericas = v_numericas.drop('Class')
v_numericas = v_numericas.tolist()
v_numericas



Index(['Clump_thickness', 'Uniformity_of_cell_size',
       'Uniformity_of_cell_shape', 'Marginal_adhesion',
       'Single_epithelial_cell_size', 'Bare_nuclei', 'Bland_chromatin',
       'Normal_nucleoli', 'Mitoses'],
      dtype='object')

In [36]:
from feature_engine import discretisation as dsc

disc = dsc.EqualFrequencyDiscretiser(q=4, variables=v_numericas)
disc.fit(data)
data1 = disc.transform(data)

In [37]:
def IV(variable, target, data):
    woe = data.groupby(variable)[target].agg([('Total', 'count'), ('A favor', 'sum')]).reset_index()
    woe['En contra'] = woe['Total'] - woe['A favor']
    woe['OR'] = woe['A favor'] / woe['En contra']
    woe['WOE'] = np.log((woe['A favor'] / np.sum(woe['A favor'])) / (woe['En contra'] / np.sum(woe['En contra'])))
    woe['Dif'] = (woe['A favor'] / np.sum(woe['A favor'])) - (woe['En contra'] / np.sum(woe['En contra']))

    iv = np.sum(woe['WOE'] * woe['Dif'])
    result = pd.DataFrame(columns=['Variable', 'IV'])
    result.loc[len(result)] = [variable, iv]
    return result

In [38]:
resultados = pd.DataFrame(columns=['Variable', 'IV'])

for var in v_numericas:
    A = IV(var, 'Class', data1)
    resultados.loc[len(resultados)] = [A.loc[0, 'Variable'], A.loc[0, 'IV']]

# Ordenar al final
resultados = resultados.sort_values('IV', ascending=False)

print(resultados)

                      Variable        IV
4  Single_epithelial_cell_size  0.026155
6              Bland_chromatin  0.025953
0              Clump_thickness  0.020526
5                  Bare_nuclei  0.018632
2     Uniformity_of_cell_shape  0.015521
1      Uniformity_of_cell_size  0.015129
7              Normal_nucleoli  0.014387
3            Marginal_adhesion  0.013034
8                      Mitoses  0.000000


In [39]:
# Excluimos las debiles
variables_utiles = [
    'Single_epithelial_cell_size',
    'Bland_chromatin',
    'Clump_thickness'
]

# Creamos nuevo DataFrame con las variables útiles + la variable objetivo 'Class'
data_filtrado = data1[variables_utiles + ['Class']]

data_filtrado.head()

,Single_epithelial_cell_size,Bland_chromatin,Clump_thickness,Class
0,0,1,2,2
1,2,1,2,2
2,0,1,1,2
3,1,1,2,2
4,0,1,1,2


In [48]:
# Convertir la variable 'Class' a binaria
data_filtrado['Class'] = data_filtrado['Class'].apply(lambda x: 1 if x == 4 else 0)

# Verificamos
data_filtrado.head()

<ipython-input-48-1fbf7d9df77a>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_filtrado['Class'] = data_filtrado['Class'].apply(lambda x: 1 if x == 4 else 0)


,Single_epithelial_cell_size,Bland_chromatin,Clump_thickness,Class,Class_binary
0,0,1,2,0,0
1,2,1,2,0,0
2,0,1,1,0,0
3,1,1,2,0,0
4,0,1,1,0,0


In [49]:
from sklearn.model_selection import train_test_split

# Separar variables predictoras
X = data_filtrado.drop('Class', axis=1)
y = data_filtrado['Class']

# Dividir en entrenamiento prueba 25%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


### b. Genere el modelo de regresión logística y evalúe la exclusión de variables mediante la significancia de los coeficientes. Además, calcule las métricas de clasificación que se implementaron en la parte práctica e interprete sus resultados más importantes.

In [50]:
# Geneamos el modelo de regresión logística
import statsmodels.api as sm

X_train_sm = sm.add_constant(X_train, prepend=True)
rlog = sm.Logit(endog=y_train, exog=X_train_sm)
rlog_result = rlog.fit()
print(rlog_result.summary())

         Current function value: 0.000000
         Iterations: 35
                           Logit Regression Results                           
Dep. Variable:                  Class   No. Observations:                  512
Model:                          Logit   Df Residuals:                      507
Method:                           MLE   Df Model:                            4
Date:                Mon, 05 May 2025   Pseudo R-squ.:                     inf
Time:                        04:43:22   Log-Likelihood:            -2.1408e-07
converged:                      False   LL-Null:                        0.0000
Covariance Type:            nonrobust   LLR p-value:                     1.000
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
const                         -28.1543   1.92e+04     -0.001      0.999   -3.76e+04    3.76e+04
Single_epithel

/usr/local/lib/python3.11/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.11/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.11/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.11/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.11/dist-packa

In [52]:
X_test = sm.add_constant(X_test, prepend=True)
prob = rlog_result.predict(exog=X_test)
prob
y_pred = np.where(prob<0.5,0,1)
y_pred

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [53]:
matriz_confusion = pd.crosstab(y_test.ravel(),y_pred,rownames=['Real'],colnames=['Prediccion'])
matriz_confusion

<ipython-input-53-e727b9c4736c>:1: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  matriz_confusion = pd.crosstab(y_test.ravel(),y_pred,rownames=['Real'],colnames=['Prediccion'])


Prediccion,0
Real,
0,171


In [54]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_true = y_test,y_pred=y_pred,normalize=True)
accuracy

1.0

In [56]:
from sklearn.metrics import roc_auc_score
roc_auc = roc_auc_score(y_test,prob)
roc_auc

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


nan

In [57]:
Gini = 2*roc_auc-1
Gini

nan

### c Genere el modelo SVM, calcule sus métricas de clasificación y compárelas con las del modelo de regresión logística para ver si hubo o no mejoras.

In [ ]:
from sklearn.svm import SVC
SVM_clf = SVC(C=100,kernel,'linear',random_state=123)
SVM_clf.fit(X_train,y_train)

In [ ]:
y_pred = SVM_clf.predict(X_test)
y_pred

In [58]:
matriz_confusion = pd.crosstab(y_test.ravel(),y_pred,
                              rownames=['Real'],colnames=['Prediccion'])
matriz_confusion

<ipython-input-58-01a29eb45674>:1: FutureWarning: Series.ravel is deprecated. The underlying array is already 1D, so ravel is not necessary.  Use `to_numpy()` for conversion to a numpy array instead.
  matriz_confusion = pd.crosstab(y_test.ravel(),y_pred,


Prediccion,0
Real,
0,171


In [59]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_true = y_test,y_pred=y_pred,normalize=True)
accuracy

1.0